In [ ]:
import sys
sys.path.append("..")
from datetime import datetime
import torch

from src.environment.utils import smooth
from src.data import load_data, load_and_align_data, PAIRS, get_field

from bokeh.palettes import Category10
import bokeh.plotting as bk
bk.output_notebook()

# Read Historical Data

In [ ]:
PAIRS = {
    'Bitcoin': 'XBTEUR',
    'Ethereum': 'ETHEUR',
    'Ripple': 'XRPEUR',
    'Cardano': 'ADAEUR'
}

In [ ]:
data, times = load_and_align_data(PAIRS, interval=5)

In [ ]:
times_ = torch.tensor([t.timestamp() for t in times])
dt = float(times_.diff().mean().round())
print(f"dt = {dt}")
prices = torch.tensor(get_field(data, 'open')).T
volume = torch.tensor(get_field(data, 'volume')).T

history = [{
    'time'  : t,
    'prices': p,
    'volume': v
} for t, p, v in zip(times_, prices, volume)]

history = sorted(history, key=lambda x: x['time'])

len(history)

In [ ]:
f1 = bk.figure(title=f"Prices", x_axis_type="datetime", x_axis_label="t", y_axis_label="USD", width=1200, height=500)

for i, (name, price) in enumerate(zip(data.keys(), prices.T)):
    r = f1.line(times[::100], price[::100], line_width=2, legend_label=f"{name}", color=Category10[10][i%10])
f1.legend.click_policy = "hide"

bk.show(f1)

In [ ]:
f1 = bk.figure(title=f"Volumes", x_axis_type="datetime", x_axis_label="t", y_axis_label="EUR", width=1200, height=500)

eur_vol = volume * prices

for i, (name, vol) in enumerate(zip(data.keys(), eur_vol.T)):
    r = f1.line(times[::10], vol[::10], line_width=2, legend_label=f"{name}", color=Category10[10][i%10])
f1.legend.click_policy = "hide"

bk.show(f1)

In [ ]:
f1 = bk.figure(title=f"Volumes", x_axis_type="datetime", x_axis_label="t", y_axis_label="EUR", width=1200, height=500)
max_len = 1000
eur_vol = (volume * prices)[:max_len]

eur_vol_rel = (eur_vol[1:] / eur_vol[:-1])

for i, (name, vol) in enumerate(zip(data.keys(), eur_vol_rel.T)):
    r = f1.line(times[1:max_len], vol, line_width=2, legend_label=f"{name}", color=Category10[10][i%10])
f1.legend.click_policy = "hide"

bk.show(f1)

In [ ]:
from src.environment.proto_v02 import MultiCurrencyEnv

base_t = 60*3
tau_p = torch.tensor([base_t*5, base_t*60, base_t*60*24, base_t*60*24*30])
print((tau_p / (3600 * 24)).tolist(), 60*24*30)

env = MultiCurrencyEnv(
    N=len(data),
    C0=1_000.0,
    tau_p=tau_p,
    sell_fee=0.05,
    buy_fee=0.05,
    # reward_mode="log",
    reward_mode="realized_roi",
    transaction_eps=1e-3,
    save_history=False,
    bankruptcy_threshold=1.0
)

# state = env.reset(history[0])
# for elem in history[:int(len(history)*0.9):]:
#     a = torch.tanh(torch.randn(env.action_size))

#     state, reward, done, info = env.step(a, data=elem)
#     if done:
#         break

In [ ]:
env.state_size, env.action_size

# Integrate Agent

In [ ]:
from src.network import VanillaNetwork, VanillaQNetwork, SquashedStochasticVanillaNetwork
from src.agent.ddpg import DDPGConfig, VanillaDDPG
from src.agent.sac import SACConfig, SACAgent
from src.agent.train import train_on_historical


agent_type = "sac"

if agent_type == "ddpg":
    agent = VanillaDDPG(
        actor=VanillaNetwork(env.state_size, env.action_size, sizes=[128, 128], act=torch.tanh, out_act=torch.tanh),
        critic=VanillaQNetwork(env.state_size, env.action_size, sizes=[128, 128], act=torch.tanh),
        config=DDPGConfig(
            gamma=0.999,
            tau=100.0,
            noise_std=0.1,
            buffer_size=1_000_000,
            device="cpu"
        )
    )

elif agent_type == "sac":
    agent = SACAgent(
        actor=SquashedStochasticVanillaNetwork(env.state_size, env.action_size, sizes=[128, 128], act=torch.tanh),
        critic=VanillaQNetwork(env.state_size, env.action_size, sizes=[128, 128], act=torch.tanh),
        config=SACConfig(
            gamma=0.999,
            tau=100.0,
            alpha = 0.1,
            target_entropy = None,
            buffer_size=10_000_000,
            device="cpu"
        )
    )

else:
    raise NotImplementedError(f"The agent type '{agent_type}' is not implemented.")

In [ ]:
agent.load(f"../data/agent/{agent_type}_{env.reward_mode}.ptm")

In [ ]:
loss, rewards, info = [], [], []

In [ ]:
agent.config.noise_std = 0.3

_loss, _rewards, _info = train_on_historical(
    agent, env, history[:int(len(history)*0.8)],
    n_episodes=100, batch_size=32, n_updates=16,
    max_steps=50000, warm_up=1000,
    actor_lr=1e-4, critic_lr=1e-4,
    optim="AdamW"
)
loss += _loss
rewards += _rewards
info += _info

In [ ]:
agent.save(f"../data/agent/{agent_type}_{env.reward_mode}.ptm")

In [ ]:
# loss_a = torch.tensor([[loss_dict['actor_loss'] for loss_dict in episode_loss] for episode_loss in loss])
loss_a = [sum([loss_dict['actor_loss'] for loss_dict in episode_loss]) / len(episode_loss) for episode_loss in loss]

fig = bk.figure(title="Actor Losses", x_axis_label="Training Iteration [Epochs]", y_axis_label="Loss", width=900, height=320)
# fig.line(torch.arange(loss_a.numel()) / loss_a.shape[1], loss_a.flatten(), line_width=2, legend_label="Loss / Batch", color=Category10[10][0], alpha=0.3)
# fig.line(torch.arange(len(loss_a)) + 0.5, loss_a.mean(1), line_width=2, legend_label="Epoch Average", color=Category10[10][0])
fig.line(list(range(len(loss_a))), loss_a, line_width=2, legend_label="Epoch Average", color=Category10[10][0])
bk.show(fig)

In [ ]:
# loss_c = torch.tensor([[loss_dict['critic_loss'] for loss_dict in episode_loss] for episode_loss in loss])
loss_c = [sum([loss_dict['critic_loss'] for loss_dict in episode_loss]) / len(episode_loss) for episode_loss in loss]

fig = bk.figure(title="Critic Losses", x_axis_label="Training Iteration [Epochs]", y_axis_label="Loss", width=900, height=320, tools="pan,wheel_zoom,box_zoom,reset,save,hover")
# fig.line(torch.arange(loss_c.numel()) / loss_c.shape[1], loss_c.flatten(), line_width=2, legend_label="Loss / Batch", color=Category10[10][2], alpha=0.3)
# fig.line(torch.arange(len(loss_c)) + 0.5, loss_c.mean(1), line_width=2, legend_label="Epoch Average", color=Category10[10][2])
fig.line(list(range(len(loss_c))), loss_c, line_width=2, legend_label="Epoch Average", color=Category10[10][2])
bk.show(fig)

In [ ]:
# rewards = torch.tensor(rewards)
rewards_ = [sum(r) / len(r) for r in rewards]

fig = bk.figure(title="Rewards", x_axis_label="Training Iteration [Episodes]", y_axis_label="Loss", width=900, height=320)
# fig.line(torch.arange(rewards.numel()) / rewards.shape[1], rewards.flatten(), line_width=2, legend_label="Reward", color=Category10[10][4], alpha=0.3)
# fig.line(torch.arange(len(rewards)) + 0.5, rewards.mean(1), line_width=2, legend_label="Average Reward / Episode", color=Category10[10][4])
fig.line(list(range(len(rewards_))), rewards_, line_width=2, legend_label="Average Reward / Episode", color=Category10[10][4])
bk.show(fig)

In [ ]:
episode = 245

ts = [datetime.fromtimestamp(item['t']) for item in info[episode]]
ps = torch.stack([item['p'] for item in info[episode]])
Vs = torch.tensor([item['V'] for item in info[episode]])
Cs = torch.tensor([item['C'] for item in info[episode]])
vs = torch.stack([item['w']*item['p'] / item['V'] for item in info[episode]])

f0 = bk.figure(title=f"Episode {episode} - Prices", x_axis_label="t", y_axis_label=r"\(p / p_{max} [1]\)", x_axis_type="datetime", width=900, height=320)
for i, name in enumerate(data):
    f0.line(ts, ps[:,i] / ps[:,i].max(), line_width=2, legend_label=f"{name}", color=Category10[10][i%10])
f0.legend.click_policy = "hide"
bk.show(f0)

f0 = bk.figure(title=f"Episode {episode} - Portfolio Fraction", x_axis_label="t", y_axis_label="EUR", x_axis_type="datetime", width=900, height=320)
f0.line(ts, Cs / Vs, line_width=2, line_dash="dashed", legend_label="Cash", color=Category10[10][0])
for i, name in enumerate(data):
    f0.line(ts, vs[:,i], line_width=2, legend_label=f"{name}", color=Category10[10][i%10])
f0.legend.click_policy = "hide"
bk.show(f0)

f1 = bk.figure(title=f"Episode {episode} - Portfolio Value", x_axis_label="t", y_axis_label="EUR", x_axis_type="datetime", width=900, height=320)
r1 = f1.line(ts, Vs, line_width=2, legend_label="Total V")
r2 = f1.line(ts, Cs, line_width=1, line_dash="dashed", legend_label="Cash C")
f1.legend.click_policy = "hide"
bk.show(f1)

fig = bk.figure(title=f"Episode {episode} - Rewards", x_axis_label="t", y_axis_label="Reward (log(V_t+1/V_t))", x_axis_type="datetime", width=900, height=320)
fig.scatter(ts, rewards[episode], size=2, color=Category10[10][4], legend_label="Reward")

returns = []
gamma = 0.999
G = 0.0
for r in reversed(rewards[episode]):
    G = r + gamma * G
    returns.insert(0, G)
fig.line(ts, returns, line_width=2, color=Category10[10][5], legend_label="Return")

hist, edges = torch.histogram(torch.tensor(rewards[episode]), bins=200, density=True)
x = (edges[:-1] + edges[1:]) / 2
figh = bk.figure(title="Reward Distribution", width=300, height=320)
figh.harea(y=x, x1=0, x2=hist, fill_color=Category10[10][4], fill_alpha=0.4)
figh.line(hist, x, line_color=Category10[10][4], line_width=2)
fig.legend.click_policy = "hide"

bk.show(bk.row(fig, figh))

In [ ]:
actions = torch.stack([item['a'] for item in info[episode]])
ts = [i for i in range(len(actions))]
skip = 10

f1 = bk.figure(title=f"Action Fraction", x_axis_label="t", y_axis_label="USD", width=900, height=320)
f1.scatter(ts[::skip], actions[::skip,0], size=1, legend_label="a_0")
for i in range(1,env.N+1):
    f1.scatter(ts[::skip], actions[::skip,i], size=1, legend_label=f"a_{i}", color=Category10[10][(i+1)%10])
f1.legend.click_policy = "hide"
bk.show(f1)

In [ ]:
raise

# Validate

In [ ]:
start = int(len(history)*0.8)

env.save_history = True

hist_s = []
hist_r = []
hist_i = []

state = env.reset(history[start])
for elem in history[start:]:
    a = agent.act(state.to_tensor(), explore=False)
    state, reward, done, _info = env.step(a, data=elem)

    hist_s.append(state)
    hist_r.append(reward)
    hist_i.append(_info)

    if done:
        break

In [ ]:
Vs = [item['V'] for item in hist_i]
Cs = [item['C'] for item in hist_i]
ts = [datetime.fromtimestamp(item["t"]) for item in hist_i]

skip = 10
f1 = bk.figure(
    title=f"Prices",
    width=1200, height=400,
    x_range=(ts[0], ts[-1]),
    x_axis_type="datetime",
    x_axis_label="t",
    y_axis_label="EUR",
)
for i, (name, price) in enumerate(zip(data.keys(), prices.T)):
    r = f1.line(times[start::skip], price[start::skip], line_width=2, legend_label=f"{name}", color=Category10[10][i%10])
f1.legend.click_policy = "hide"
bk.show(f1)

fig1 = bk.figure(
    title=f"Portfolio Value",
    width=1200, height=400,
    x_range=(ts[0], ts[-1]),
    x_axis_type="datetime",
    x_axis_label="t",
    y_axis_label="EUR",
)
fig1.line(ts, Vs, line_width=2, legend_label="Total V")
fig1.line(ts, Cs, line_width=1, line_dash="dashed", legend_label="Cash C")
fig1.legend.click_policy = "hide"
bk.show(fig1)

fig2 = bk.figure(
    title=f"Rewards",
    width=1200, height=400,
    x_range=(ts[0], ts[-1]),
    x_axis_type="datetime",
    x_axis_label="t",
    y_axis_label="Reward (log(V_t+1/V_t))",
)
fig2.line(ts, hist_r, line_width=2, color=Category10[10][4])

hist, edges = torch.histogram(torch.tensor(hist_r), bins=200, density=True)
x = (edges[:-1] + edges[1:]) / 2

figh = bk.figure(title="Reward Distribution", width=300, height=400)
figh.harea(y=x, x1=0, x2=hist, fill_color=Category10[10][4], fill_alpha=0.4)
figh.line(hist, x, line_color=Category10[10][4], line_width=2)

bk.show(bk.row(fig2, figh))

In [ ]:
for i, name in enumerate(data):
    print(i, name)

In [ ]:
skip = 100
Vs = torch.tensor([item["V"] for item in hist_i[::skip]])
Cs = torch.tensor([item["C"] for item in hist_i[::skip]])
ws = torch.stack([item["w"] for item in hist_i[::skip]])
ps = torch.stack([item["p"] for item in hist_i[::skip]])
vs = (ps * ws / Vs[:,None])

print(Vs.shape)
print(Cs.shape)
print(ws.shape)
print(ps.shape)
print(vs.shape)

f = bk.figure(
    title=f"Portfolio Fration",
    width=1200, height=400,
    x_range=(ts[0], ts[-1]),
    x_axis_type="datetime",
    x_axis_label=r"\(\text{t}\)",
    y_axis_label=r"\(\text{Fraction} [1]\)",
)
f.line(ts[::skip], Cs / Vs, line_width=2, line_dash="dashed", legend_label="Cash", color=Category10[10][0])
for i, name in enumerate(data):
    f.line(ts[::skip], vs[:,i], line_width=2, legend_label=f"{name}", color=Category10[10][i%10])
f.legend.click_policy = "hide"
bk.show(f)

skip = 10
f1 = bk.figure(
    title=f"Prices",
    width=1200, height=400,
    x_range=(ts[0], ts[-1]),
    x_axis_type="datetime",
    x_axis_label=r"\(\text{t}\)",
    y_axis_label=r"\(p / p_{max} [1]\)",
)
for i, (name, price) in enumerate(zip(data.keys(), prices.T)):
    r = f1.line(times[start::skip], price[start::skip] / price.max(), line_width=2, legend_label=f"{name}", color=Category10[10][i%10])
f1.legend.click_policy = "hide"
bk.show(f1)